In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim , col

In [0]:
df = spark.table("workspace.bronze.prd_info")

In [0]:
df = spark.table("workspace.bronze.prd_info2")

In [0]:
if df is None:
    df = spark.table("workspace.bronze.prd_info")

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date"
}

In [0]:
for (old_name, new_name) in RENAME_MAP.items():
    if new_name not in df.columns:
        df = df.withColumnRenamed(old_name, new_name)

In [0]:
display(df)

product_id,product_key,product_name,product_cost,product_line,product_start_date,product_end_date
210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,null,R,2003-07-01,null
211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,null,R,2003-07-01,null
212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12,S,2011-07-01,2007-12-28
213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14,S,2012-07-01,2008-12-27
214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13,S,2013-07-01,null
215,AC-HE-HL-U509,Sport-100 Helmet- Black,12,S,2011-07-01,2007-12-28
216,AC-HE-HL-U509,Sport-100 Helmet- Black,14,S,2012-07-01,2008-12-27
217,AC-HE-HL-U509,Sport-100 Helmet- Black,13,S,2013-07-01,null
218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3,M,2011-07-01,2007-12-28
219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3,M,2011-07-01,2007-12-28


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.bronze.prd_info2")

df_new = df.select(
    F.col("prd_id"),
    F.substring(F.col("prd_key"), 1, 5).alias("prd_key")
)

In [0]:
(
    df_new.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.sliver.prd_info_short")
)

In [0]:
%sql
UPDATE workspace.bronze.prd_info
SET prd_key = SUBSTRING(prd_key, 7);

num_affected_rows
397


In [0]:
(
    df.write
      .mode("overwrite")
      .format("delta")
      .option("overwriteSchema", "true")
      .saveAsTable("workspace.sliver.prod_info")
)

In [0]:
from pyspark.sql import functions as F

df = (
    spark.table("workspace.sliver.prod_info").alias("prod1")
    .join(
        spark.table("workspace.sliver.prd_info_short").alias("prod2"),
        F.col("prod1.prd_id") == F.col("prod2.prd_id"),
        "left"
    )
    .select("prod1.*", F.col("prod2.prd_key").alias("category_key"))
)

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.sliver.prod_info")
)

In [0]:
%sql
select * from workspace.sliver.prod_info

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt,category_key
210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,null,R,2003-07-01,null,CO-RF
211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,null,R,2003-07-01,null,CO-RF
212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12,S,2011-07-01,2007-12-28,AC-HE
213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14,S,2012-07-01,2008-12-27,AC-HE
214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13,S,2013-07-01,null,AC-HE
215,AC-HE-HL-U509,Sport-100 Helmet- Black,12,S,2011-07-01,2007-12-28,AC-HE
216,AC-HE-HL-U509,Sport-100 Helmet- Black,14,S,2012-07-01,2008-12-27,AC-HE
217,AC-HE-HL-U509,Sport-100 Helmet- Black,13,S,2013-07-01,null,AC-HE
218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3,M,2011-07-01,2007-12-28,CL-SO
219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3,M,2011-07-01,2007-12-28,CL-SO
